In [3]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from torchvision.utils import make_grid
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import umap
from itertools import product

# ==============================================================================
# 1. Cấu hình và Thiết lập (Configuration and Setup)
# ==============================================================================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SAVE_DIR = "results_wae_annealing_v4"
os.makedirs(SAVE_DIR, exist_ok=True)

config = {
    "latent_dim": 32,
    "n_classes": 10,
    "batch_size": 128,
    "epochs": 50,  # Tăng lại số epoch để mô hình có đủ thời gian hội tụ sau annealing
    "lr": 1e-3,
    "rho_prior": 0.6,  # Giữ nguyên rho_prior để tránh ảnh hưởng đến kết quả
    "epsilon": 1e-8,
    # --- Cấu hình cho Annealing ---
    "recon_weight": 1.0,
    "sup_mmd_weight": 20.0,
    "unsup_mmd_weight": 50.0,
    "anneal_epochs": 25,
}

# ==============================================================================
# 2. Các hàm tiện ích
# ==============================================================================
def sample_uniform_sphere(n_samples, dim, device=DEVICE):
    eta = torch.randn(n_samples, dim, device=device)
    return F.normalize(eta, p=2, dim=1)

def mobius_reparam(eps, mu, rho):
    rho = rho.unsqueeze(-1) if rho.dim() == 1 else rho
    eps_mu_dot = torch.sum(eps * mu, dim=1, keepdim=True)
    numerator = (1 - rho**2) * eps + 2 * rho**2 * mu + 2 * rho * eps_mu_dot * mu
    denominator = 1 + 2 * rho * eps_mu_dot + rho**2
    z = numerator / (denominator + config["epsilon"])
    return F.normalize(z, p=2, dim=1)

def rbf_kernel(x, y, sigma):
    dist_sq = 2 - 2 * (x @ y.t())
    return torch.exp(-dist_sq / (2 * sigma**2 + config["epsilon"]))

def mmd_loss(q_samples, p_samples, sigma=None):
    if q_samples.shape[0] < 2 or p_samples.shape[0] < 2:
        return torch.tensor(0.0, device=DEVICE)
    if sigma is None:
        with torch.no_grad():
            dists = torch.pdist(torch.cat([q_samples, p_samples], dim=0))
            sigma = dists.median()
    k_qq = rbf_kernel(q_samples, q_samples, sigma).mean()
    k_pp = rbf_kernel(p_samples, p_samples, sigma).mean()
    k_qp = rbf_kernel(q_samples, p_samples, sigma).mean()
    return k_qq + k_pp - 2 * k_qp

# ==============================================================================
# 3. Kiến trúc Mô hình
# ==============================================================================
class EncoderCNN(nn.Module):
    def __init__(self, latent_dim):
        super(EncoderCNN, self).__init__()
        self.conv_block = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=4, stride=2, padding=1), nn.BatchNorm2d(32), nn.ReLU(True),
            nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1), nn.BatchNorm2d(64), nn.ReLU(True),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1), nn.BatchNorm2d(128), nn.ReLU(True),
        )
        self.fc_block = nn.Sequential(nn.Flatten(), nn.Linear(128 * 4 * 4, 256), nn.ReLU(True))
        self.fc_mu = nn.Linear(256, latent_dim)
        self.fc_s = nn.Linear(256, 1)
    def forward(self, x):
        x = self.conv_block(x)
        x = self.fc_block(x)
        return self.fc_mu(x), self.fc_s(x)

class DecoderCNN(nn.Module):
    def __init__(self, latent_dim):
        super(DecoderCNN, self).__init__()
        self.fc_block = nn.Sequential(nn.Linear(latent_dim, 256), nn.ReLU(True), nn.Linear(256, 128 * 4 * 4), nn.ReLU(True))
        self.deconv_block = nn.Sequential(
            nn.ConvTranspose2d(128, 64, kernel_size=3, stride=2, padding=1), nn.BatchNorm2d(64), nn.ReLU(True),
            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1), nn.BatchNorm2d(32), nn.ReLU(True),
            nn.ConvTranspose2d(32, 1, kernel_size=4, stride=2, padding=1), nn.Sigmoid()
        )
    def forward(self, z):
        x = self.fc_block(z)
        x = x.view(-1, 128, 4, 4)
        return self.deconv_block(x)

class SphericalWAE_Supervised(nn.Module):
    def __init__(self, latent_dim, n_classes):
        super(SphericalWAE_Supervised, self).__init__()
        self.encoder = EncoderCNN(latent_dim)
        self.decoder = DecoderCNN(latent_dim)
        self.prior_mus = nn.Parameter(torch.randn(n_classes, latent_dim))
        self.rho_p = config["rho_prior"]
    def encode_to_distribution(self, x):
        mu_q_unnormalized, s_q = self.encoder(x)
        mu_q = F.normalize(mu_q_unnormalized, p=2, dim=1)
        rho_q = torch.sigmoid(s_q).squeeze(-1) * (1 - config["epsilon"])
        return mu_q, rho_q
    def forward(self, x):
        mu_q, rho_q = self.encode_to_distribution(x)
        eps = sample_uniform_sphere(x.shape[0], config["latent_dim"], device=x.device)
        z_q = mobius_reparam(eps, mu_q, rho_q)
        x_hat = self.decoder(z_q)
        return x_hat, z_q

# ==============================================================================
# 4. Hàm tính toán Mất mát
# ==============================================================================
def calculate_loss(x, y, x_hat, z_q, model, sup_mmd_weight, unsup_mmd_weight):
    recon_loss = F.binary_cross_entropy(x_hat, x, reduction='mean')

    supervised_mmd_loss = 0.0
    normalized_prior_mus = F.normalize(model.prior_mus, p=2, dim=1)
    for c in range(config["n_classes"]):
        class_mask = (y == c)
        if class_mask.sum() > 1:
            supervised_mmd_loss += mmd_loss(z_q[class_mask],
                                            mobius_reparam(sample_uniform_sphere(class_mask.sum(), config["latent_dim"]),
                                                           normalized_prior_mus[c].expand(class_mask.sum(), -1),
                                                           torch.full((class_mask.sum(),), model.rho_p, device=DEVICE)))
    supervised_mmd_loss /= config["n_classes"]

    random_classes = torch.randint(0, config["n_classes"], (x.size(0),), device=DEVICE)
    z_p_unsupervised = mobius_reparam(sample_uniform_sphere(x.size(0), config["latent_dim"]),
                                      normalized_prior_mus[random_classes],
                                      torch.full((x.size(0),), model.rho_p, device=DEVICE))
    unsupervised_mmd_loss = mmd_loss(z_q, z_p_unsupervised)

    total_loss = (config["recon_weight"] * recon_loss) + \
                 (sup_mmd_weight * supervised_mmd_loss) + \
                 (unsup_mmd_weight * unsupervised_mmd_loss)

    return total_loss, recon_loss, supervised_mmd_loss, unsupervised_mmd_loss

# ==============================================================================
# 5. Vòng lặp Huấn luyện
# ==============================================================================
def train_epoch(model, train_loader, optimizer, epoch, scheduler):
    model.train()
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{config['epochs']}")
    loss_acc, recon_acc, sup_mmd_acc, unsup_mmd_acc = 0.0, 0.0, 0.0, 0.0

    anneal_rate = min(1.0, (epoch + 1) / config["anneal_epochs"]) # Bắt đầu từ epoch 0
    current_sup_weight = config["sup_mmd_weight"] * anneal_rate
    current_unsup_weight = config["unsup_mmd_weight"] * anneal_rate

    for data, labels in pbar:
        data, labels = data.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()

        x_hat, z_q = model(data)

        loss, recon, sup_mmd, unsup_mmd = calculate_loss(data, labels, x_hat, z_q, model, current_sup_weight, current_unsup_weight)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        loss_acc += loss.item()
        recon_acc += recon.item()
        sup_mmd_acc += sup_mmd.item()
        unsup_mmd_acc += unsup_mmd.item()

        pbar.set_postfix({
            "Loss": f"{loss.item():.3f}", "Recon": f"{recon.item():.3f}",
            "SupMMD": f"{sup_mmd.item():.4f}", "UnsupMMD": f"{unsup_mmd.item():.4f}",
            "λ": f"{current_sup_weight:.2f}", "γ": f"{current_unsup_weight:.2f}"
        })

    scheduler.step()
    n_batches = len(train_loader)
    print(f"====> Epoch {epoch+1} Avg Loss: Total={loss_acc/n_batches:.4f}, Recon={recon_acc/n_batches:.4f}, SupMMD={sup_mmd_acc/n_batches:.4f}, UnsupMMD={unsup_mmd_acc/n_batches:.4f}")
    return loss_acc/n_batches, recon_acc/n_batches, sup_mmd_acc/n_batches, unsup_mmd_acc/n_batches

# ==============================================================================
# 6. Trực quan hóa và Hàm chính
# ==============================================================================
def plot_random_samples_from_priors(model, save_dir="."):
    print("Bắt đầu sinh ảnh ngẫu nhiên từ các tiên nghiệm...")
    model.eval()
    n_classes, latent_dim = config["n_classes"], config["latent_dim"]
    fig, axes = plt.subplots(n_classes, 8, figsize=(8 * 1.5, n_classes * 1.5))
    with torch.no_grad():
        normalized_prior_mus = F.normalize(model.prior_mus, p=2, dim=1)
        for c in range(n_classes):
            mu_p = normalized_prior_mus[c].expand(8, -1)
            rho_p = torch.full((8,), model.rho_p, device=DEVICE)
            eps = sample_uniform_sphere(8, latent_dim, device=DEVICE)
            z_p = mobius_reparam(eps, mu_p, rho_p)
            generated_images = model.decoder(z_p)
            for i, img in enumerate(generated_images):
                ax = axes[c, i]
                ax.imshow(img.cpu().squeeze(), cmap='gray')
                ax.axis('off')
                if i == 0:
                    ax.text(-10, 14, f'{c}', verticalalignment='center', horizontalalignment='right', fontsize=12, fontweight='bold')
    plt.suptitle("Ảnh sinh ngẫu nhiên từ các thành phần Tiên nghiệm (Annealing)")
    plt.savefig(f'{save_dir}/random_samples_from_priors.png')
    plt.close(fig)

def from_latent(net, vec):
    with torch.no_grad():
        net.eval()
        vec_tensor = torch.from_numpy(vec).unsqueeze(0).to(DEVICE).float()
        vec_tensor_normalized = F.normalize(vec_tensor, p=2, dim=1)
        return net.decoder(vec_tensor_normalized).cpu().numpy().reshape(28, 28)

def get_sampling_grid(net, grid):
    base = torch.randn(config["latent_dim"] - 2)
    image_list = [torch.from_numpy(from_latent(net, np.hstack([vec_2d, base.numpy()]))) for vec_2d in grid]
    return torch.stack(image_list).unsqueeze(1)

def plot_grid_samples(model, save_dir="."):
    print("Bắt đầu sinh ảnh từ lưới phẳng (phương pháp so sánh)...")
    grid_points = list(product(np.linspace(-1.5, 1.5, 8), np.linspace(-1.5, 1.5, 8)))
    results = get_sampling_grid(model, grid_points)
    fig = plt.figure(figsize=(10, 10))
    img_grid = make_grid(results, nrow=8)
    plt.imshow(img_grid.permute(1, 2, 0))
    plt.title("Ảnh sinh ra từ Lưới Phẳng (Grid Sampling)")
    plt.axis('off')
    plt.savefig(f'{save_dir}/grid_samples.png')
    plt.close(fig)

def plot_results(history, model, test_loader, save_dir="."):
    print("Bắt đầu vẽ biểu đồ và trực quan hóa kết quả...")
    fig = plt.figure(figsize=(12, 8))
    plt.plot([h[0] for h in history], label='Total Loss')
    plt.plot([h[1] for h in history], label='Reconstruction Loss')
    plt.plot([h[2] for h in history], label='Supervised MMD Loss')
    plt.plot([h[3] for h in history], label='Unsupervised MMD Loss')
    plt.title('Lịch sử Huấn luyện (WAE with Annealing)')
    plt.xlabel('Epoch'); plt.ylabel('Loss Value'); plt.legend(); plt.grid(True)
    plt.savefig(f'{save_dir}/loss_history.png'); plt.close(fig)

    model.eval()
    with torch.no_grad():
        data, _ = next(iter(test_loader)); data = data.to(DEVICE); x_hat, _ = model(data)
        fig = plt.figure(figsize=(20, 4))
        n = 10
        for i in range(n):
            ax = plt.subplot(2, n, i + 1); plt.imshow(data[i].cpu().squeeze(), cmap='gray'); plt.title("Gốc"); ax.axis('off')
            ax = plt.subplot(2, n, i + 1 + n); plt.imshow(x_hat[i].cpu().squeeze(), cmap='gray'); plt.title("Tái tạo"); ax.axis('off')
        plt.savefig(f'{save_dir}/reconstructions.png'); plt.close(fig)

        print("Bắt đầu chiếu không gian ẩn bằng UMAP...")
        latent_mus, labels = [], []
        for data, target in tqdm(test_loader, desc="Encoding test set for UMAP"):
            mu_q, _ = model.encode_to_distribution(data.to(DEVICE))
            latent_mus.append(mu_q.cpu().numpy()); labels.append(target.numpy())

        latent_mus = np.concatenate(latent_mus, axis=0); labels = np.concatenate(labels, axis=0)

        reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, n_components=2, random_state=42)
        embedding = reducer.fit_transform(latent_mus)

        prior_mus_np = F.normalize(model.prior_mus, p=2, dim=1).cpu().detach().numpy()
        prior_embedding = reducer.transform(prior_mus_np)

        fig = plt.figure(figsize=(12, 10))
        scatter = plt.scatter(embedding[:, 0], embedding[:, 1], c=labels, cmap='Spectral', s=5, alpha=0.7)
        plt.scatter(prior_embedding[:, 0], prior_embedding[:, 1], c=range(10), cmap='Spectral', marker='*', s=500, edgecolor='black', label='Prior Centers')
        plt.title('Không gian ẩn (Latent Space) - UMAP (Annealing)')
        plt.legend(handles=scatter.legend_elements(num=10)[0], labels=list(range(10)))
        plt.colorbar(scatter); plt.savefig(f'{save_dir}/latent_space_umap.png'); plt.close(fig)

    plot_random_samples_from_priors(model, save_dir=save_dir)
    plot_grid_samples(model, save_dir=save_dir)


In [4]:
transform = transforms.Compose([transforms.ToTensor()])
train_dataset = datasets.MNIST('./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST('./data', train=False, transform=transform)

# Sử dụng num_workers > 0 để tăng tốc độ load dữ liệu nếu có thể
num_workers = 2 if os.name == 'nt' else 4
train_loader = DataLoader(train_dataset, batch_size=config["batch_size"], shuffle=True, pin_memory=True, num_workers=num_workers)
test_loader = DataLoader(test_dataset, batch_size=config["batch_size"], shuffle=False, pin_memory=True, num_workers=num_workers)

model = SphericalWAE_Supervised(latent_dim=config["latent_dim"], n_classes=config["n_classes"]).to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=config["lr"])
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=30, gamma=0.5)

In [5]:
print(f"Bắt đầu huấn luyện trên thiết bị: {DEVICE} với kiến trúc WAE và Annealing")
print(f"Cấu hình: {config}")

history = []
for epoch in range(config["epochs"]):
    avg_losses = train_epoch(model, train_loader, optimizer, epoch, scheduler)
    history.append(avg_losses)

print("Hoàn tất huấn luyện!")

torch.save(model.state_dict(), f'{SAVE_DIR}/spcauchy_wae_annealing.pth')
print(f"Đã lưu mô hình đã huấn luyện vào '{SAVE_DIR}/spcauchy_wae_annealing.pth'")

plot_results(history, model, test_loader, save_dir=SAVE_DIR)

Bắt đầu huấn luyện trên thiết bị: cuda với kiến trúc WAE và Annealing
Cấu hình: {'latent_dim': 32, 'n_classes': 10, 'batch_size': 128, 'epochs': 50, 'lr': 0.001, 'rho_prior': 0.6, 'epsilon': 1e-08, 'recon_weight': 1.0, 'sup_mmd_weight': 20.0, 'unsup_mmd_weight': 50.0, 'anneal_epochs': 25}


Epoch 1/50: 100%|██████████| 469/469 [00:31<00:00, 14.94it/s, Loss=0.214, Recon=0.125, SupMMD=0.0897, UnsupMMD=0.0087, λ=0.80, γ=2.00]


====> Epoch 1 Avg Loss: Total=0.2926, Recon=0.1860, SupMMD=0.1094, UnsupMMD=0.0096


Epoch 2/50: 100%|██████████| 469/469 [00:29<00:00, 15.91it/s, Loss=0.323, Recon=0.113, SupMMD=0.1069, UnsupMMD=0.0097, λ=1.60, γ=4.00]


====> Epoch 2 Avg Loss: Total=0.2820, Recon=0.1198, SupMMD=0.0811, UnsupMMD=0.0081


...

====> Epoch 49 Avg Loss: Total=1.7542, Recon=0.0966, SupMMD=0.0698, UnsupMMD=0.0052


Epoch 50/50: 100%|██████████| 469/469 [00:28<00:00, 16.24it/s, Loss=2.822, Recon=0.095, SupMMD=0.1235, UnsupMMD=0.0051, λ=20.00, γ=50.00]


====> Epoch 50 Avg Loss: Total=1.7567, Recon=0.0966, SupMMD=0.0703, UnsupMMD=0.0051
Hoàn tất huấn luyện!
Đã lưu mô hình đã huấn luyện vào 'results_wae_annealing_v4/spcauchy_wae_annealing.pth'
Bắt đầu vẽ biểu đồ và trực quan hóa kết quả...
Bắt đầu chiếu không gian ẩn bằng UMAP...


Encoding test set for UMAP: 100%|██████████| 79/79 [00:01<00:00, 48.13it/s]
/usr/local/lib/python3.11/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Bắt đầu sinh ảnh ngẫu nhiên từ các tiên nghiệm...
Bắt đầu sinh ảnh từ lưới phẳng (phương pháp so sánh)...


In [6]:
# ==============================================================================
# Sửa lỗi và Bổ sung hàm vẽ Slerp
# ==============================================================================

def slerp(p0, p1, t, epsilon=1e-8):
    """
    Nội suy trên mặt cầu (Spherical Linear Interpolation).
    p0, p1: các vector bắt đầu và kết thúc (đã được chuẩn hóa).
    t: một giá trị hoặc tensor chứa các giá trị từ 0 đến 1.
    """
    # Tính góc giữa hai vector
    omega = torch.acos(torch.dot(p0, p1).clamp(-1, 1))
    sin_omega = torch.sin(omega)

    # Nếu hai vector quá gần nhau, trả về vector ban đầu để tránh chia cho 0
    if sin_omega.item() < epsilon:
        # Mở rộng p0 để có cùng số chiều với output mong muốn khi t là một tensor
        return p0.unsqueeze(0).expand(len(t), -1)

    # Đảm bảo t có cùng device với các vector
    t = t.to(p0.device)

    # Công thức Slerp
    a = torch.sin((1.0 - t) * omega) / sin_omega
    b = torch.sin(t * omega) / sin_omega

    # unsqueeze() để thực hiện phép nhân broadcast đúng cách
    return a.unsqueeze(-1) * p0.unsqueeze(0) + b.unsqueeze(-1) * p1.unsqueeze(0)


def plot_slerp(model, save_dir=".", num_steps=10):
    """
    Vẽ và lưu ảnh được tạo ra từ phép nội suy Slerp giữa các cặp tiên nghiệm.
    """
    print("Bắt đầu sinh ảnh nội suy Slerp...")
    model.eval()

    # Chọn một vài cặp chữ số thú vị để nội suy
    interpolation_pairs = [(1, 7), (3, 5), (2, 8), (4, 9)]
    num_pairs = len(interpolation_pairs)

    fig, axes = plt.subplots(num_pairs, num_steps, figsize=(num_steps * 1.5, num_pairs * 1.5))

    with torch.no_grad():
        t_values = torch.linspace(0, 1, num_steps)

        for row, (digit_a, digit_b) in enumerate(interpolation_pairs):
            # LẤY VECTOR VÀ SỬA LỖI: Thêm `dim=0`
            mu_a = F.normalize(model.prior_mus[digit_a].detach(), dim=0)
            mu_b = F.normalize(model.prior_mus[digit_b].detach(), dim=0)

            # Thực hiện Slerp
            interpolated_z = slerp(mu_a, mu_b, t_values).to(DEVICE)

            # Giải mã các vector z trung gian
            generated_images = model.decoder(interpolated_z)

            # Vẽ các ảnh
            for col, img in enumerate(generated_images):
                ax = axes[row, col]
                ax.imshow(img.cpu().squeeze(), cmap='gray')
                ax.axis('off')

                # Ghi nhãn cho ảnh đầu và cuối
                if col == 0:
                    ax.set_title(f'{digit_a}')
                if col == num_steps - 1:
                    ax.set_title(f'{digit_b}')

    plt.suptitle("Nội suy Slerp giữa các Tiên nghiệm (Slerp Interpolation)")
    plt.tight_layout(rect=[0, 0.03, 1, 0.95]) # Điều chỉnh layout để title không bị đè
    plt.savefig(f'{save_dir}/slerp_interpolation.png')
    plt.close(fig)
    print(f"Đã lưu ảnh Slerp vào '{save_dir}/slerp_interpolation.png'")

In [7]:
# THÊM LỆNH GỌI HÀM MỚI Ở ĐÂY
plot_slerp(model, save_dir=SAVE_DIR)

Bắt đầu sinh ảnh nội suy Slerp...
Đã lưu ảnh Slerp vào 'results_wae_annealing_v4/slerp_interpolation.png'


---